In [2]:
# Librerías principales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib.pyplot as plt
import altair as als
import seaborn as sns
import folium

# 1. Limpieza de datos

### Cargar el Dataframe en Colab
Durante este paso se realiza el cargue del dataset de wage11 bajo el nombre de data_source

In [3]:
from google.colab import files
uploaded = files.upload()

Saving dataset.xlsx to dataset.xlsx


In [6]:
# Propósito: ubicar la ruta efectiva del dataset y validar su existencia antes de cargarlo.
df = pd.read_excel("dataset.xlsx")
df.head()

,ID,Genero,Etnia,Municipio,Departamento,Region,Lote,cohorte,Estrato,Fecha de Nacimiento,...,Cabeza de familia,Temas de interés,Sistemas Operativos,Estado,Puntaje,Nivel,Fecha Realización,Institución,Grupo,Fecha de registro
0,1,Hombre,Ninguna de las anteriores,SANTA MARTA,MAGDALENA,Región 6,Lote 1,NaN,Estrato 3,1990-07-13 00:00:00,...,NaN,Encontrar un nuevo empleo\nEmprender nuevos re...,Windows,Importado,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Hombre,"Negro(a), Mulato(a), Afrodescendiente, Afrocol...",SABANALARGA,ATLÁNTICO,Región 6,Lote 1,NaN,Estrato 1,2005-12-10 00:00:00,...,NaN,Encontrar un nuevo empleo\nEmprender nuevos re...,Windows,Importado,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Hombre,Ninguna de las anteriores,BARRANQUILLA,ATLÁNTICO,Región 6,Lote 1,NaN,Estrato 3,1992-02-17 00:00:00,...,NaN,Encontrar un nuevo empleo\nEmprender nuevos re...,Windows,Pre-Inscrito,93.33,1.0,2024-10-17 16:22:57,NaN,NaN,2024-10-15 03:52:02
3,4,Hombre,Ninguna de las anteriores,SANTA MARTA,MAGDALENA,Región 6,Lote 1,NaN,Estrato 2,1996-04-28 00:00:00,...,NaN,Fortalecer mis conocimientos,Windows,Importado,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Hombre,Ninguna de las anteriores,SANTA MARTA,MAGDALENA,Región 6,Lote 1,1.0,Estrato 1,1998-12-02 00:00:00,...,NaN,Encontrar un nuevo empleo\nEmprender nuevos re...,Windows,Certificado,46.67,1.0,2024-10-24 08:00:00,NaN,R6-L1-PG-B-V-JN-G186,2024-10-24 08:00:00


### Normalización de nombres de columnas
En este paso, realizamos una normalizacion a los nombres de las columnas, con el fin de utilizar nombres mas descriptivos de acuerdo al dato que estan proporcionando

In [7]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('¿', '')
df.columns = df.columns.str.replace('?', '')
df.columns = df.columns.str.replace('á', 'a')
df.columns = df.columns.str.replace('é', 'e')
df.columns = df.columns.str.replace('í', 'i')
df.columns = df.columns.str.replace('ó', 'o')
df.columns = df.columns.str.replace('ú', 'u')

df.columns

Index(['id', 'genero', 'etnia', 'municipio', 'departamento', 'region', 'lote',
       'cohorte', 'estrato', 'fecha_de_nacimiento', 'discapacidad',
       'nivel_academico', 'programa', 'jornada', 'modalidad', 'es_campesino',
       'victima_del_conflicto_armado', 'tiene_computadora', 'tiene_internet',
       'tiene_14_horas_a_la_semana',
       'motivo_por_el_que_no_dispone_de_14_horas', 'como_se_entero',
       'trabaja_actualmente', 'empresa_donde_trabaja', 'cargo_que_desempeña',
       'salario', 'actividad_actual', 'rol_actual', 'cabeza_de_familia',
       'temas_de_interes', 'sistemas_operativos', 'estado', 'puntaje', 'nivel',
       'fecha_realizacion', 'institucion', 'grupo', 'fecha_de_registro'],
      dtype='str')

### Eliminacion de Duplicados

#### Eliminación de duplicados por id.

Primero se realiza una busqueda de duplicados a traves del subset definido. Cuyo unico valor esta dado por el id.

In [8]:
duplicados_totales = df.duplicated(subset=['id']).sum()
print("Duplicados encontrados:", duplicados_totales)

df_cleaned = df.drop_duplicates(subset=['id'])
df_cleaned.describe()

Duplicados encontrados: 1411


,id,cohorte,cabeza_de_familia,puntaje,nivel
count,79877.000000,10360.000000,2896.0,41410.000000,41335.000000
mean,44426.214617,5.146042,1.0,57.350581,1.024725
std,25174.525865,4.272367,0.0,30.220254,0.184350
min,1.000000,1.000000,1.0,0.000000,1.000000
25%,22677.000000,1.000000,1.0,33.330000,1.000000
50%,45366.000000,3.000000,1.0,60.000000,1.000000
75%,66228.000000,9.000000,1.0,83.330000,1.000000
max,86461.000000,14.000000,1.0,100.000000,3.000000


### Eliminacion de duplicados por estructura
Posteriomente se realiza una busqueda por el resto de valores del dataframe. Donde se comparan todas las columnas a excepcion del id, para la identificacion de duplicados estructurales (Donde todos los valores son exactamente iguales)

In [9]:
index = [
  'genero',
  'etnia',
  'municipio',
  'departamento',
  'region',
  'lote',
  'cohorte',
  'estrato',
  'fecha_de_nacimiento',
  'discapacidad',
  'nivel_academico',
  'programa',
  'jornada',
  'modalidad',
  'es_campesino',
  'victima_del_conflicto_armado',
  'tiene_computadora',
  'tiene_internet',
  'tiene_14_horas_a_la_semana',
  'motivo_por_el_que_no_dispone_de_14_horas',
  'como_se_entero',
  'trabaja_actualmente',
  'empresa_donde_trabaja',
  'cargo_que_desempeña',
  'salario',
  'actividad_actual',
  'rol_actual',
  'cabeza_de_familia',
  'temas_de_interes',
  'sistemas_operativos',
  'estado',
  'puntaje',
  'nivel',
  'fecha_realizacion',
  'institucion',
  'grupo',
  'fecha_de_registro'
]

duplicados_estructura = df_cleaned.duplicated(
  subset=index
).sum()

print("Duplicados por estructura:", duplicados_estructura)

df_cleaned = df_cleaned.drop_duplicates(subset=index)
df_cleaned.describe()

Duplicados por estructura: 1534


,id,cohorte,cabeza_de_familia,puntaje,nivel
count,78343.000000,9706.000000,2887.0,41273.000000,41198.000000
mean,44622.611759,5.235318,1.0,57.540948,1.024807
std,25325.235720,4.398521,0.0,30.088887,0.184651
min,1.000000,1.000000,1.0,0.000000,1.000000
25%,22851.500000,1.000000,1.0,33.330000,1.000000
50%,45594.000000,3.000000,1.0,60.000000,1.000000
75%,66611.500000,10.000000,1.0,83.330000,1.000000
max,86461.000000,14.000000,1.0,100.000000,3.000000


### Detección de valores faltantes
Ahora que ya se realizo la eliminacion de valores duplicados, se identifican los valores faltantes de cada una de las columnas del dataframe.

Para ello se crea un nuevo dataframe que contiene la informacion del total de valores nulos y el porcetaje que representa para cada una de las columnas

In [10]:
valores_nulos = df_cleaned.isnull().sum()
porcentaje_nulos = (df_cleaned.isnull().sum() / len(df_cleaned)) * 100

tabla_nulos = pd.DataFrame({
    'cantidad_nulos': valores_nulos,
    'porcentaje_%': porcentaje_nulos
})

tabla_nulos.sort_values(by='cantidad_nulos', ascending=False)

,cantidad_nulos,porcentaje_%
motivo_por_el_que_no_dispone_de_14_horas,78311,99.959154
cabeza_de_familia,75456,96.314923
cohorte,68637,87.610891
rol_actual,63325,80.830451
empresa_donde_trabaja,54866,70.033060
salario,54692,69.810959
grupo,54503,69.569713
cargo_que_desempeña,54158,69.129341
institucion,50155,64.019759
nivel,37145,47.413298


In [11]:
df_cleaned.columns

Index(['id', 'genero', 'etnia', 'municipio', 'departamento', 'region', 'lote',
       'cohorte', 'estrato', 'fecha_de_nacimiento', 'discapacidad',
       'nivel_academico', 'programa', 'jornada', 'modalidad', 'es_campesino',
       'victima_del_conflicto_armado', 'tiene_computadora', 'tiene_internet',
       'tiene_14_horas_a_la_semana',
       'motivo_por_el_que_no_dispone_de_14_horas', 'como_se_entero',
       'trabaja_actualmente', 'empresa_donde_trabaja', 'cargo_que_desempeña',
       'salario', 'actividad_actual', 'rol_actual', 'cabeza_de_familia',
       'temas_de_interes', 'sistemas_operativos', 'estado', 'puntaje', 'nivel',
       'fecha_realizacion', 'institucion', 'grupo', 'fecha_de_registro'],
      dtype='str')

## Validación de Formatos y Consistencia
Posteriormente se realiza la validacion de formatos y consistencia de cada una de las columnas de dataframe.

Donde se realizan validaciones de columnas mayores a 0, columnas booleanas y columnas dadas por enumeradores (Como la columna de categoria)

In [14]:
def validar_y_normalizar(df):

  df_clean = df.copy()
  inconsistencias = {}


  columnas_mayor_0 = [
    'id', 'puntaje', 'nivel'
  ]


  for col in columnas_mayor_0:
    if col in df_clean.columns:

      df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

      df_clean[col] = df_clean[col].fillna(0)

      invalidos = df_clean[df_clean[col] < 0]
      inconsistencias[col] = len(invalidos)


  columnas_booleanas = [
    'cabeza_de_familia', 'es_campesino',
    'victima_del_conflicto_armado', 'tiene_computadora', 'tiene_internet',
    'tiene_14_horas_a_la_semana', 'trabaja_actualmente'
  ]


  for col in columnas_booleanas:
    if col in df_clean.columns:
      temp_series = df_clean[col].astype(str).str.lower().str.strip()

      temp_series = temp_series.str.replace('sí', 'si').str.replace('¿', '').str.replace('?', '')

      boolean_mapping = {'si': True, 'no': False, '1': True, '0': False, 1: True, 0: False}

      mapped_values = temp_series.map(boolean_mapping)

      # Use direct column assignment so pandas can change dtype when needed.
      df_clean[col] = mapped_values.where(mapped_values.notna(), None).astype(object)

      invalidos = df_clean[col][~df_clean[col].isin([True, False, None])]
      inconsistencias[col] = len(invalidos)


  reporte = pd.DataFrame({
    "Variable": inconsistencias.keys(),
    "Registros_Invalidos": inconsistencias.values()
  }).sort_values("Registros_Invalidos", ascending=False)

  return df_clean, reporte


df_validado, reporte_validacion = validar_y_normalizar(df_cleaned)

df_limpio = df_validado.copy()

reporte_validacion

,Variable,Registros_Invalidos
0,id,0
1,puntaje,0
2,nivel,0
3,cabeza_de_familia,0
4,es_campesino,0
5,victima_del_conflicto_armado,0
6,tiene_computadora,0
7,tiene_internet,0
8,tiene_14_horas_a_la_semana,0
9,trabaja_actualmente,0


In [15]:
pd.set_option('display.max_columns', None)
df_limpio.sample(10)

,id,genero,etnia,municipio,departamento,region,lote,cohorte,estrato,fecha_de_nacimiento,discapacidad,nivel_academico,programa,jornada,modalidad,es_campesino,victima_del_conflicto_armado,tiene_computadora,tiene_internet,tiene_14_horas_a_la_semana,motivo_por_el_que_no_dispone_de_14_horas,como_se_entero,trabaja_actualmente,empresa_donde_trabaja,cargo_que_desempeña,salario,actividad_actual,rol_actual,cabeza_de_familia,temas_de_interes,sistemas_operativos,estado,puntaje,nivel,fecha_realizacion,institucion,grupo,fecha_de_registro
79109,76040,Mujer,Ninguna de las anteriores,SANTIAGO DE CALI,VALLE DEL CAUCA,Región 9,Lote 2,NaN,Estrato 3,2005-08-23,Sin discapacidad,No registra/No aplica,Inteligencia Artificial,Jornada mañana,Presencial,False,False,True,True,True,NaN,Amigos o familiares,False,NaN,NaN,NaN,Otro,Egresado,None,Adquirir conocimientos nuevos,Windows,Certificado,91.67,1.0,2025-08-31 12:31:43,Fundación Maria Cano,R9-L2-VAL-IA-B-P-JM-G2445,2025-08-31 11:28:34
61621,12749,Hombre,Ninguna de las anteriores,JAMUNDÍ,VALLE DEL CAUCA,Región 9,Lote 2,1.0,Estrato 5,1976-11-15,Sin discapacidad,Posgrado,Análisis de Datos,Jornada nocturna,Virtual,False,False,True,True,True,NaN,Universidad Libre,True,SOS,gerente,Mayor a 9.000.000,Trabajar,NaN,None,Emprender nuevos retos,Windows,Retirado,0.00,0.0,NaN,NaN,NaN,2024-08-15 13:02:07
41008,66153,Mujer,Ninguna de las anteriores,RIOHACHA,LA GUAJIRA,Región 6,Lote 1,NaN,Estrato 2,1987-03-13 00:00:00,Sin discapacidad,No registra/No aplica,Inteligencia Artificial,Jornada nocturna,Virtual,False,False,True,True,True,NaN,MINTIC,True,Secretaría de educación de Riohacha,Docente de aula,entre 3.000.000 y 5.000.000,Otro,Docente,None,Adquirir conocimientos nuevos,Windows,Certificado,50.00,1.0,2025-07-14 18:43:52,Convenio IA Para funcionarios y docentes La Gu...,R6-L1-IA-B-V-JN-G2353,2025-07-14 18:30:36
47394,83219,Mujer,Ninguna de las anteriores,VALLEDUPAR,CESAR,Región 6,Lote 1,NaN,Estrato 3,1991-05-29 00:00:00,Sin discapacidad,No registra/No aplica,Inteligencia Artificial,Jornada mañana,Presencial,False,False,True,True,True,NaN,MINTIC,False,NaN,NaN,NaN,Otro,Servidor Publico,None,Adquirir conocimientos nuevos,Windows,Formado,0.00,1.0,2025-11-21 09:42:48,Convenio UPC,R6-L1-CES-IA-B-P-JM-G3385,2025-11-21 09:34:21
51198,68402,Hombre,Ninguna de las anteriores,LA CALERA,CUNDINAMARCA,Región 7,Lote 2,NaN,Estrato 2,1989-12-29,Sin discapacidad,No registra/No aplica,Inteligencia Artificial,Jornada nocturna,Virtual,False,False,True,True,True,NaN,MINTIC,True,Construcciones Santa Sofia,Asesor,entre 1.000.000 y 3.000.000,Otro,Otros,None,Adquirir conocimientos nuevos,Windows,Certificado,58.33,1.0,2025-07-21 21:39:28,NaN,R7-L2-IA-B-V-JN-G3001,2025-07-21 21:27:18
80128,82399,Hombre,Ninguna de las anteriores,SANTIAGO DE CALI,VALLE DEL CAUCA,Región 9,Lote 2,NaN,Estrato 3,1984-04-06,Sin discapacidad,No registra/No aplica,Inteligencia Artificial,Jornada nocturna,Virtual,False,False,True,True,True,NaN,"Redes Sociales (Facebook, Twitter, Instagram, ...",True,Construcción,Contratista,entre 1.000.000 y 3.000.000,Otro,Otros,None,Adquirir conocimientos nuevos,Windows,Pre-Inscrito,0.00,0.0,NaN,NaN,NaN,2025-11-02 15:01:36
68108,35219,Hombre,Ninguna de las anteriores,BUENAVENTURA,VALLE DEL CAUCA,Región 9,Lote 2,NaN,Sin estratificar,1984-01-20,Sin discapacidad,No registra/No aplica,Inteligencia Artificial,Jornada nocturna,Presencial,False,False,True,True,True,NaN,MINTIC,False,NaN,NaN,NaN,Otro,NaN,None,Adquirir conocimientos nuevos,Windows,Retirado,33.33,1.0,2025-02-27 15:36:40,Convenio FADP,NaN,2024-10-30 08:10:11
64376,18554,Mujer,NaN,SANTIAGO DE CALI,VALLE DEL CAUCA,Región 9,Lote 2,NaN,NaN,2006-02-04,NaN,Profesional universitario,NaN,NaN,NaN,False,False,False,False,False,NaN,Universidad Libre,False,NaN,NaN,NaN,NaN,NaN,None,Adquirir conocimientos nuevos,Windows,Importado,0.00,0.0,NaN,Unilibre Cali,NaN,NaN
19403,32860,Hombre,"Negro(a), Mulato(a), Afrodescendiente, Afrocol...",SAN JUAN DEL CESAR,LA GUAJIRA,Región 6,Lote 1,NaN,Estrato 1,1999-05